# 02 · Feature Engineering

Criamos variáveis derivadas que ajudam os modelos a capturar padrões,
e exportamos o dataset processado para `data/processed/`.


## Setup


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.data.loader import load_raw, split
from src.features.engineering import build_features, get_feature_names


## 1. Carrega Dados Brutos


In [ ]:
# Carrega do disco se já foi baixado no notebook 01; senão, baixa da URL
raw_path = '../data/raw/creditcard.csv'
df_raw = load_raw(raw_path if os.path.exists(raw_path) else None)


## 2. Aplica Feature Engineering

Todas as transformações ficam em `src/features/engineering.py` — reutilizável na API.


In [ ]:
df = build_features(df_raw)

# Mostra as novas colunas criadas
novas = [c for c in df.columns if c not in df_raw.columns]
print(f'Novas features criadas: {novas}')
df[novas + ['Class']].describe().round(3)


## 3. Valida as Features


In [ ]:
# Checa valores nulos
nulos = df.isnull().sum()
print('Valores nulos por coluna:')
print(nulos[nulos > 0] if nulos.any() else '  Nenhum ✅')

# Checa que Is_night é binário
assert df['Is_night'].isin([0, 1]).all(), 'Is_night deve ser 0 ou 1'

# Checa que Hour está no intervalo esperado
assert df['Hour'].between(0, 23).all(), 'Hour deve estar entre 0 e 23'

# Checa que Amount_log >= 0 (log1p nunca é negativo)
assert (df['Amount_log'] >= 0).all(), 'Amount_log deve ser >= 0'

print('\nTodas as validações passaram ✅')


## 4. Divisão Treino/Teste


In [ ]:
X_train, X_test, y_train, y_test = split(df)

feature_names = get_feature_names(df)
print(f'\nFeatures ({len(feature_names)}): {feature_names}')


## 5. Exporta Dados Processados


In [ ]:
os.makedirs('../data/processed', exist_ok=True)

X_train.to_csv('../data/processed/X_train.csv', index=False)
X_test.to_csv('../data/processed/X_test.csv',   index=False)
y_train.to_csv('../data/processed/y_train.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv',   index=False)

import json
with open('../data/processed/feature_names.json', 'w') as f:
    json.dump(feature_names, f)

print('Dados processados salvos em data/processed/')
